# Class 15 experiments 5 and 6

Load the two newest Tracker exports and plot their measured paths in the rotating frame. Positions are in centimeters and time is in seconds. The initial-velocity arrows use a constrained quadratic fit to the first six points.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

data_files = {
    "Experiment 5": Path("Class15_experiment_5.txt"),
    "Experiment 6": Path("Class15_experiment_6.txt"),
}

trajectories = {}
for label, path in data_files.items():
    data = np.loadtxt(path, skiprows=2)
    trajectories[label] = {
        "path": path,
        "t": data[:, 0],
        "x": data[:, 1],
        "y": data[:, 2],
    }
    print(
        f"{label}: {len(data)} points, "
        f"t = {data[0, 0]:.3f} to {data[-1, 0]:.3f} s"
    )


## Estimate the initial conditions

For each coordinate $q=x',y'$, fit the first six measurements to

$$q(\tau)-q(0)=v_{q,0}\tau+\tfrac12a_{q,0}\tau^2, \qquad \tau=t-t_0,$$

while fixing the initial position to the first measured point.


In [ ]:
def estimate_initial_conditions(t, x, y, n_fit=6):
    n_fit = min(n_fit, len(t))
    tau = t - t[0]
    design = np.column_stack((tau[1:n_fit], 0.5 * tau[1:n_fit]**2))
    vx0, ax0 = np.linalg.lstsq(
        design, x[1:n_fit] - x[0], rcond=None
    )[0]
    vy0, ay0 = np.linalg.lstsq(
        design, y[1:n_fit] - y[0], rcond=None
    )[0]
    return {
        "x0": x[0], "y0": y[0],
        "vx0": vx0, "vy0": vy0,
        "ax0": ax0, "ay0": ay0,
        "n_fit": n_fit, "tau_max": tau[n_fit - 1],
    }

for label, tr in trajectories.items():
    tr["initial"] = estimate_initial_conditions(tr["t"], tr["x"], tr["y"])
    ic = tr["initial"]
    print(f"\n{label}")
    print(f"  (x0, y0)   = ({ic['x0']:.3f}, {ic['y0']:.3f}) cm")
    print(f"  (vx0, vy0) = ({ic['vx0']:.3f}, {ic['vy0']:.3f}) cm/s")
    print(f"  fit window  = {ic['n_fit']} points through {ic['tau_max']:.3f} s")


## Circular path plots

Both panels use equal horizontal and vertical scales. Color indicates time; the white marker is the first point and the square is the final point. The arrow shows the fitted initial velocity multiplied by the displayed time scale so it can be drawn in position units.


In [ ]:
def add_circular_grid(ax, radius_max):
    theta = np.linspace(0, 2 * np.pi, 500)
    for radius in np.arange(1, np.ceil(radius_max) + 1):
        ax.plot(
            radius * np.cos(theta), radius * np.sin(theta),
            color="0.86", linewidth=0.8, zorder=0,
        )
    for angle in np.deg2rad(np.arange(0, 360, 30)):
        ax.plot(
            [0, radius_max * np.cos(angle)],
            [0, radius_max * np.sin(angle)],
            color="0.91", linewidth=0.7, zorder=0,
        )
    ax.axhline(0, color="0.65", linewidth=0.8, zorder=0)
    ax.axvline(0, color="0.65", linewidth=0.8, zorder=0)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel(r"$x'$ (cm)")
    ax.set_ylabel(r"$y'$ (cm)")

arrow_dt_s = 0.08
fig, axes = plt.subplots(1, 2, figsize=(13, 6), constrained_layout=True)

for ax, (label, tr) in zip(axes, trajectories.items()):
    t, x, y = tr["t"], tr["x"], tr["y"]
    ic = tr["initial"]
    radius_max = 1.12 * np.hypot(x, y).max()
    add_circular_grid(ax, radius_max)
    points = ax.scatter(
        x, y, c=t, cmap="viridis", s=65,
        edgecolor="black", linewidth=0.5, zorder=3,
    )
    ax.scatter(x[0], y[0], s=140, facecolor="white", edgecolor="black", zorder=4, label="start")
    ax.scatter(x[-1], y[-1], s=110, marker="s", facecolor="#ef5350", edgecolor="black", zorder=4, label="end")
    ax.quiver(
        x[0], y[0], ic["vx0"] * arrow_dt_s, ic["vy0"] * arrow_dt_s,
        angles="xy", scale_units="xy", scale=1, color="#d32f2f",
        width=0.012, zorder=5, label=fr"$\vec{{v}}'_0 \times {arrow_dt_s:.2f}\,\mathrm{{s}}$",
    )
    ax.set_title(label)
    ax.legend(frameon=False, loc="best")
    fig.colorbar(points, ax=ax, label="time (s)", shrink=0.82)

fig.suptitle("Measured rotating-frame paths")
plt.show()


## Direct overlay

This final plot puts both measured paths on one common set of axes. Use the side-by-side plots above when time coloring is important, and this overlay for a quick geometric comparison.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
all_radius = max(np.hypot(tr["x"], tr["y"]).max() for tr in trajectories.values())
add_circular_grid(ax, 1.12 * all_radius)

for label, tr in trajectories.items():
    ax.plot(tr["x"], tr["y"], "o-", linewidth=1.8, markersize=5, label=label)
    ax.scatter(tr["x"][0], tr["y"][0], s=100, facecolor="white", edgecolor="black", zorder=4)
    ax.scatter(tr["x"][-1], tr["y"][-1], s=90, marker="s", edgecolor="black", zorder=4)

ax.set_title("Experiments 5 and 6: measured-path overlay")
ax.legend(frameon=False)
plt.show()
